<a href="https://colab.research.google.com/github/EshikaAbbaraju/Role_Specified_Collective_Foraging_Task_Model/blob/main/Role_Specified_Collective_Foraging_Task_Model_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!git clone https://github.com/JerryGuo2001/Role_Speficifed_Collective_Foraging_Task.git
#%cd Role_Speficifed_Collective_Foraging_Task

#tune the danger from aliens
#print out snapshot labels at the top

%cd /content/Role_Speficifed_Collective_Foraging_Task
CSV_DIR = "gridworld/"

Cloning into 'Role_Speficifed_Collective_Foraging_Task'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 305 (delta 21), reused 35 (delta 11), pack-reused 255 (from 1)
Receiving objects: 100% (305/305), 1.84 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Role_Speficifed_Collective_Foraging_Task


In [53]:
# -*- coding: utf-8 -*-
"""
Collective foraging + security: agents keep positions and world state (mines, visited, etc.).
A “round” is exactly one agent’s block of `moves_per_round` steps; agents **alternate** each round:
round 1 → forager moves 5× (security frozen), round 2 → security moves 5× (forager frozen), then repeat.
Only one agent moves per timestep. GIF includes an initial frame with both visible.

Leader / follower (two different notions — one per agent)
--------------------------------------------------------
**Forager** “leader” vs “follower” is *not* who moves first in the schedule; turns stay fixed
(forager on even rounds, security on odd). It only changes *how the forager chooses moves* when
it is the forager’s turn:

- **Forager as leader** (`role_mode="leader"`): neighbor scoring uses **positive** λ
  (`_lambda_for_moves`): higher value for cells **farther** from security (spread out / “lead”).
  The explore–exploit mix `w` is bumped by +0.2 (capped at 1), so more weight on **exploration**
  (`_w_identity_adjust`).
- **Forager as follower** (`role_mode="follower"`): λ is **negative**: prefers cells **closer**
  to security. `w` is reduced by 0.2 (floored at 0) → more **exploitation**.

**Security** “leader” vs “follower” controls *which movement branch* runs on role-move beats;
separate chase beats use **belief** vs a threshold (no direct alien visibility — see ``SecurityAgent``):

- **Security as leader**: always uses the **lead** branch — step toward the forager’s **predicted**
  next cell (linear extrapolation from `prev_pos` → current forager position).
- **Security as follower**: always uses the **follow** branch — `_orbit_near` the forager within
  `follow_radius` (Manhattan), i.e. stay close and patrol nearby.

**Security as auto** (default): `Vmove_lead_t` vs `Vmove_follow_t` and `Vtype_t` pick the branch each
step; `lead_weight` / `follow_weight` adapt over a short window.

How to run “forager = leader, security = follower”
--------------------------------------------------
Call `run_foraging_simulation(..., forager_identity="leader", security_identity="follower")`.
Aliases like `"l"` / `"f"` work; see `_normalize_identity`.

**Important:** `forager_identity` and `security_identity` are applied *after* you build
`ForagerConfig` / `SecurityConfig`: they **overwrite** each config’s `role_mode` via
`dataclasses.replace`. To set roles only in code without those kwargs, pass
`ForagerConfig(..., role_mode="leader")` and `SecurityConfig(..., role_mode="follower")` *and* use
the default identities `"auto"`, or call `replace` yourself before constructing agents.
"""
from __future__ import annotations

from collections import deque
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Set, Tuple

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# Prefer bundled grid CSVs under ./gridworld when present; else current directory.
CSV_DIR = "gridworld/" if os.path.isdir("gridworld") else "."


def _normalize_identity(s: Optional[str]) -> str:
    """
    Map CLI-style strings onto internal role tokens: 'leader', 'follower', or 'auto'.

    Accepts (case-insensitive): for leader use lead, leader, or l; for follower use follow,
    follower, or f; for auto use auto, a, empty string, model, or formula. None → auto.

    Returned values are stored on ``ForagerConfig.role_mode`` and ``SecurityConfig.role_mode``
    (after normalization the code also accepts 'lead' / 'follow' in lowercased checks).
    """
    if s is None:
        return "auto"
    x = str(s).strip().lower()
    if x in ("lead", "leader", "l"):
        return "leader"
    if x in ("follow", "follower", "f"):
        return "follower"
    if x in ("auto", "a", "", "model", "formula"):
        return "auto"
    raise ValueError(f"identity must be 'leader', 'follower', or 'auto', got {s!r}")


# =============================================================================
# Environment
# =============================================================================
def load_env_from_csv(
    csv_path: str,
    prob_A: float = 0.8,
    prob_B: float = 0.5,
    prob_C: float = 0.2,
) -> pd.DataFrame:
    """
    Build a grid DataFrame indexed by (Row, Col): mine type, per-step reward probability,
    alien hazard level, and a human-readable Location key.
    """
    raw = pd.read_csv(csv_path)
    raw = raw.rename(columns={"x": "Row", "y": "Col", "mine_type": "mine_type", "alien_id": "alien_id"})
    rows = []
    for _, row in raw.iterrows():
        r, c = int(row["Row"]), int(row["Col"])
        mine_type = str(row["mine_type"]).strip() if not pd.isna(row["mine_type"]) else ""
        alien_val = row["alien_id"]

        # Map mine labels to dig success rate (used as immediate reward when digging).
        if mine_type == "Gold Mine A":
            reward_prob, richness = prob_A, "high"
        elif mine_type == "Gold Mine B":
            reward_prob, richness = prob_B, "medium"
        elif mine_type == "Gold Mine C":
            reward_prob, richness = prob_C, "low"
        else:
            mine_type, reward_prob, richness = "", 0.0, "none"

        alien_level = 0 if (pd.isna(alien_val) or str(alien_val).strip() == "") else int(float(alien_val))

        rows.append({
            "Row": r, "Col": c, "Location": f"{r}:{c}",
            "mine_type": mine_type, "richness": richness,
            "reward_prob": float(reward_prob), "alien_level": alien_level,
        })
    env = pd.DataFrame(rows)
    env.set_index(["Row", "Col"], inplace=True, drop=False)
    return env


# =============================================================================
# Forager
# =============================================================================
@dataclass
class ForagerConfig:
    moves_per_round: int = 5  # steps per “round” for whichever agent is active
    lambda_sec: float = 1.0  # scales security-distance term in neighbor scoring
    w_scale: float = 1.0  # logistic steepness for explore vs exploit weight w_t
    move_cost: float = 0.0  # subtracted from move value in _move_to_best_A
    mine_capacity_high: int = 8
    mine_capacity_medium: int = 5
    mine_capacity_low: int = 2
    security_pos: Optional[Tuple[int, int]] = None  # default: grid center (see agent __init__)
    seed: Optional[int] = None
    # role_mode (forager semantics only):
    #   "auto"   — use signed lambda_sec as given; no extra shift on w_t.
    #   "leader" — |lambda| in neighbor score so distance to security *increases* value of A_goal;
    #              w += 0.2 (more explore). Good for “forager leads / ranges ahead”.
    #   "follower" — negative |lambda| so *closer* to security is better; w -= 0.2 (more exploit).
    # Easiest preset: pass forager_identity=... into run_foraging_simulation (overwrites this field).
    role_mode: str = "auto"


class ForagerAgent:
    def __init__(self, env: pd.DataFrame, cfg: ForagerConfig):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)

        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self._start_pos = (max_r // 2, max_c // 2)
        self.pos = self._start_pos
        self.security_pos = cfg.security_pos or (max_r // 2, max_c // 2)

        self.t = 0  # forager step counter (used in Vmove denominator)
        self.total_reward = 0.0
        self.round_reward = 0.0  # cumulative reward in current forager segment (reset between blocks)
        self.visited: Set[Tuple[int, int]] = {self.pos}
        self.stunned_steps = 0  # alien stun: skip decisions for this many steps

        self.log: List[dict] = []
        self.frames_action: List[str] = []  # used by security for stun-rate stats
        self.prev_pos: Optional[Tuple[int, int]] = None  # position before last step (for security lead)
        self.current_round: int = 0

        # Per-cell remaining digs; depleted cells yield 0 reward_prob via current_reward_prob.
        self.mine_remaining: Dict[Tuple[int, int], int] = {}
        self.initial_mine_remaining: Dict[Tuple[int, int], int] = {}
        for (r, c), row in self.env.iterrows():
            mt = str(row["mine_type"]).strip()
            if mt == "Gold Mine A":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_high)
            elif mt == "Gold Mine B":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_medium)
            elif mt == "Gold Mine C":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_low)
        self.mine_remaining = dict(self.initial_mine_remaining)

        # Alien cells “cleared” by security chase-at-spot (ground truth); set by simulation from
        # SecurityAgent.alien_cells_cleared. Used so stuns use CSV alien_id without exposing it to policies.
        self.security_cleared_alien_cells: Set[Tuple[int, int]] = set()

    def current_reward_prob(self) -> float:
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] <= 0:
            return 0.0
        return float(self.env.loc[self.pos, "reward_prob"])

    def neighbors(self) -> List[Tuple[int, int]]:
        r, c = self.pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def Vdig(self) -> float:
        return self.current_reward_prob()

    def Vmove(self) -> float:
        # Running average reward rate this episode (segment reward / steps so far).
        return self.round_reward / float(self.t) if self.t > 0 else 0.0

    def w_t(self, vd: float, vm: float) -> float:
        # Logistic: high vd vs vm → w toward 1 (more explore); low → toward 0 (more exploit).
        delta = vd - vm
        return float(1.0 / (1.0 + np.exp(-max(self.cfg.w_scale, 1e-6) * delta)))

    def E_exploit(self, s_prime: Tuple[int, int]) -> float:
        known = [p for p in self.visited if p in self.env.index and self.env.loc[p, "mine_type"] != ""]
        if not known:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in known)
        return max(float(self.env.loc[p, "reward_prob"]) for p in known) / (1.0 + d_min)

    def E_explore(self, s_prime: Tuple[int, int]) -> float:
        unv = [p for p in self.env.index if p not in self.visited]
        if not unv:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in unv)
        return 1.0 / (1.0 + d_min)

    def A_goal(self, s_prime: Tuple[int, int], w: float) -> float:
        return (1.0 - w) * self.E_exploit(s_prime) + w * self.E_explore(s_prime)

    def _lambda_for_moves(self) -> float:
        """
        Multiplier on (Manhattan distance to security) × A_goal in `_move_to_best_A`.

        - auto: return cfg.lambda_sec unchanged (you control sign and magnitude).
        - leader: positive magnitude → farther from security is better (all else equal).
        - follower: negative magnitude → nearer to security is better.

        So “forager leader + security follower” is: forager_identity="leader" (this path) and
        security_identity="follower" (security’s `_orbit_near`, unrelated to this λ).
        """
        lam = float(self.cfg.lambda_sec)
        rm = str(self.cfg.role_mode).lower()
        if rm in ("follow", "follower"):
            return -abs(lam) if lam != 0 else -1.0
        if rm in ("lead", "leader"):
            return abs(lam) if lam != 0 else 1.0
        return lam

    def _w_identity_adjust(self, w: float) -> float:
        # Shifts the logistic explore weight: leader → +0.2 (cap 1), follower → −0.2 (floor 0).
        # Combined with _lambda_for_moves, “leader” forager both ranges from security and explores more.
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            return float(min(1.0, w + 0.2))
        if rm in ("follow", "follower"):
            return float(max(0.0, w - 0.2))
        return w

    def _log(self, **kw):
        kw.setdefault("step", self.t)
        kw.setdefault("row", self.pos[0])
        kw.setdefault("col", self.pos[1])
        kw.setdefault("total_reward", self.total_reward)
        kw.setdefault("round_reward", self.round_reward)
        kw.setdefault("round", self.current_round)
        kw.setdefault("forager_role_mode", self.cfg.role_mode)
        self.log.append(kw)

    def reset_segment_reward(self):
        """Only clears per-segment reward for display; does not move agents or reset mines."""
        self.round_reward = 0.0

    def _dig(self) -> float:
        r = self.current_reward_prob()
        self.total_reward += r
        self.round_reward += r
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] > 0:
            self.mine_remaining[self.pos] -= 1
        return r

    def _move_to_best_A(self, w: float) -> Tuple[bool, int]:
        nbrs = self.neighbors()
        if not nbrs:
            return False, 0
        best_val, best_p = -np.inf, None
        lam = self._lambda_for_moves()
        for p in nbrs:
            a = self.A_goal(p, w)
            d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
            v = lam * float(d) * a - self.cfg.move_cost
            if v > best_val:
                best_val, best_p = v, p
        if best_p is None:
            return False, 0
        self.pos = best_p
        self.visited.add(self.pos)
        al = int(self.env.loc[self.pos, "alien_level"])
        # Ground truth from CSV: stun if alien cell and escort did not clear threat within radius
        # beforehand (see SecurityAgent.alien_cells_cleared). Co-located with security is safe.
        if (
            al > 0
            and self.pos != self.security_pos
            and self.pos not in self.security_cleared_alien_cells
        ):
            self.stunned_steps = 3
        return True, al

    def step(self) -> str:
        """Returns action string for this step."""
        pos_before = tuple(self.pos)
        if self.stunned_steps > 0:
            self._log(action="stunned", decision="stunned", Vdig=np.nan, Vmove=np.nan, w=np.nan)
            self.frames_action.append("stunned")
            self.stunned_steps -= 1
            self.t += 1
            self.prev_pos = pos_before
            return "stunned"

        # Standing on a mine with remaining capacity: always dig first.
        if self.current_reward_prob() > 0.0:
            vd, vm = self.Vdig(), self.Vmove()
            w = self._w_identity_adjust(self.w_t(vd, vm))
            r_gained = self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w, r_gained=r_gained)
            self.frames_action.append("dig")
            self.t += 1
            self.prev_pos = pos_before
            return "dig"

        vd, vm = self.Vdig(), self.Vmove()
        w = self._w_identity_adjust(self.w_t(vd, vm))
        # If neighbors have reward or world unexplored, don’t “dig” on empty ground when vd>vm.
        force_move = (
            self.current_reward_prob() == 0.0
            and (
                any(self.env.loc[p, "reward_prob"] > 0 for p in self.neighbors())
                or any(p not in self.visited for p in self.env.index)
            )
        )
        if (vd > vm) and not force_move:
            self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w)
            self.frames_action.append("dig")
        else:
            moved, alien_lvl = self._move_to_best_A(w)
            self._log(
                action="move" if moved else "no_move",
                decision="move",
                Vdig=vd, Vmove=vm, w=w,
                alien_level=alien_lvl if alien_lvl else None,
            )
            self.frames_action.append("move" if moved else "no_move")
        self.t += 1
        self.prev_pos = pos_before
        return self.frames_action[-1]


# =============================================================================
# Security
# =============================================================================
@dataclass
class SecurityConfig:
    vtype_threshold: float = 0.5  # Vtype above → role_by_vtype prefers lead
    gamma: float = 0.15  # learning rate for lead/follow weight adjustment
    lead_weight_init: float = 0.5
    follow_weight_init: float = 0.5
    window_size: int = 10  # rolling window for comparing Vmove_lead vs Vmove_follow
    e_scan: float = 1.0  # scales post-move scan value Vscan
    follow_radius: int = 2  # Manhattan radius for orbit_near follower behavior
    strategy_if_tie: str = "follow"  # when vmove_lead_t == vmove_follow_t before epsilon check
    seed: Optional[int] = None
    # Participant-style “hidden alien” chase: no alien_id in policy; use belief vs threshold.
    # Keep this **low** for precautionary behavior: chase-at-spot only consumes one beat, has no
    # explicit penalty if wrong, and clears threats when mine/reward cues raise suspicion—so runs
    # tend toward **role → chase → role → chase** with frequent clearings. Raise toward ~0.45+ if you
    # want chase beats to fall through to role-style moves more often on low-belief cells.
    chase_belief_threshold: float = 0.38
    chase_clear_radius: int = 3  # ground truth: clearance applies to alien cells within this dist
    # role_mode (security semantics only):
    #   "auto"     — final_role from Vmove_lead vs Vmove_follow (and Vtype on tie); movement is only
    #                lead / follow; alternate beats add chase_at_spot when belief clears threshold.
    #   "leader"   — force final_role="lead": predict-ahead toward extrapolated forager.
    #   "follower" — force final_role="follow": orbit_near forager within follow_radius.
    # Logs set role_forced=True when leader/follower is pinned by config.
    # Easiest preset: security_identity="follower" in run_foraging_simulation.
    role_mode: str = "auto"


class SecurityAgent:
    """
    **Movement (observable):** Security does **not** see ``alien_id`` / ``alien_level``. It only
    moves as **lead** (predict forager’s next cell) or **follow** (``_orbit_near``), per ``final_role``
    and formulas (``Vmove_*``, ``Vtype_t``, etc.). ``p_alien_belief_at`` uses mine/reward cues only.

    **Beats:** Primitive steps **alternate**: (1) **role move** — one lead/follow grid step; (2)
    **chase attempt** — if ``_alien_belief_at(self.pos) >= chase_belief_threshold``, log
    ``chase_at_spot`` (no move) as “I think there’s a hidden alien here”; else the same one-step
    **lead** or **follow** move as in (1). Default threshold is low so suspicion from reward cues
    usually triggers a precautionary clear (cheap one-step beat). Then repeat.

    **Stun rescue:** If the forager still has alien ``stunned_steps > 0``, beats are **suspended**:
    every security step is one greedy step toward the forager (``move_rescue_stunned`` /
    ``rescue_stunned_no_step``) until the forager can act again; the lead/follow/chase alternation
    does not advance while rescuing.

    **Ground truth (not for policy):** On ``chase_at_spot``, any CSV alien cell within
    ``chase_clear_radius`` (Manhattan) of security is added to ``alien_cells_cleared`` so the forager
    stun rule can test “chased away beforehand” without showing aliens to the escort policy.

    **Anti ping-pong:** ``exclude`` / tie-breaking on ``_step_toward`` and ``_orbit_near`` (see below).

    **Scan:** ``Vscan`` after each step from belief at the (possibly same) cell.
    """

    def __init__(self, env, cfg: SecurityConfig, start_pos: Tuple[int, int]):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        self._start_pos = start_pos
        self.pos = start_pos

        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.Dmax = max(1, max_r + max_c)  # for normalizing forager–security distance

        self.lead_weight = float(cfg.lead_weight_init)
        self.follow_weight = float(cfg.follow_weight_init)
        self.vmove_lead_hist: deque = deque(maxlen=cfg.window_size)
        self.vmove_follow_hist: deque = deque(maxlen=cfg.window_size)

        self.log: List[dict] = []
        self._prev_security_pos: Optional[Tuple[int, int]] = None
        # True → next primitive step is lead/follow; False → chase attempt (at-spot or same role move).
        self._next_beat_is_role_move: bool = True
        # Alien cells cleared for stun logic (ground truth from env); updated on chase_at_spot only.
        self.alien_cells_cleared: Set[Tuple[int, int]] = set()

    def _neighbors(self, pos: Tuple[int, int]) -> List[Tuple[int, int]]:
        r, c = pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def _manhattan(self, a: Tuple[int, int], b: Tuple[int, int]) -> int:
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _step_toward(
        self,
        target: Tuple[int, int],
        exclude: Optional[Tuple[int, int]] = None,
        forbid_cell: Optional[Tuple[int, int]] = None,
    ) -> bool:
        """
        Greedy one-step move toward ``target``. Among neighbors minimizing Manhattan distance to
        ``target``, drop ``exclude`` only if another cell ties for that minimum (avoids pointless
        U-turns without forcing a longer step and A↔B ping-pong).
        ``forbid_cell`` drops a neighbor entirely (e.g. never step onto the forager while closing
        distance in follow/orbit).
        """
        if self.pos == target:
            return False
        nbrs = self._neighbors(self.pos)
        if forbid_cell is not None:
            nbrs = [p for p in nbrs if p != forbid_cell]
        if not nbrs:
            return False
        best_d = min(self._manhattan(p, target) for p in nbrs)
        best = [p for p in nbrs if self._manhattan(p, target) == best_d]
        if exclude is not None and len(best) > 1:
            alt = [p for p in best if p != exclude]
            if alt:
                best = alt
        self.pos = min(best, key=lambda p: (p[0], p[1]))
        return True

    def _orbit_cross_turn(self, target: Tuple[int, int], p: Tuple[int, int]) -> int:
        """2D cross product (target→self) × (self→p); sign picks a consistent tangential direction."""
        tr, tc = target
        sr, sc = self.pos
        pr, pc = p
        rx, ry = sr - tr, sc - tc
        sx, sy = pr - sr, pc - sc
        return rx * sy - ry * sx

    def _orbit_near(
        self,
        target: Tuple[int, int],
        radius: int,
        exclude: Optional[Tuple[int, int]] = None,
    ) -> bool:
        """
        Follower patrol within ``radius`` of ``target``; outside radius, ``_step_toward`` (never
        onto ``target``). In-radius: stay in the escort ring — do not stand on ``target``, and when
        directly beside ``target`` prefer sideways (orbit) steps over retracing along the corridor.
        """
        raw = self._neighbors(self.pos)
        if not raw:
            return False
        # Do not co-occupy the forager's cell; orbit around it.
        raw = [p for p in raw if p != target]
        if not raw:
            return False
        if self._manhattan(self.pos, target) > radius:
            return self._step_toward(target, exclude=exclude, forbid_cell=target)

        cands = [p for p in raw if exclude is None or p != exclude]
        if not cands:
            cands = list(raw)

        in_ring = [p for p in cands if self._manhattan(p, target) <= radius]
        pool = in_ring if in_ring else cands

        sd = self._manhattan(self.pos, target)

        def tier(p: Tuple[int, int]) -> int:
            d = self._manhattan(p, target)
            if sd == 1:
                # Beside escort: L1 "sideways" cells are d==2; stepping away along the corridor is d>=3.
                if d == 2:
                    return 0
                if d >= 3:
                    return 1
                return 2
            return abs(d - 1)

        def sort_key(p: Tuple[int, int]) -> Tuple:
            t = tier(p)
            cr = self._orbit_cross_turn(target, p)
            # Prefer lower tier, then consistent CCW tangential motion, then stable tie-break.
            return (t, -cr, p[0], p[1])

        self.pos = min(pool, key=sort_key)
        return True

    def _alien_belief_at(self, pos: Tuple[int, int]) -> float:
        """
        Participant-visible suspicion of hidden threat in [0, 1]. Does **not** read ``alien_id`` /
        ``alien_level`` (those are hidden). Uses only observable grid cues (here: mine reward prob).
        """
        base_reward = float(self.env.loc[pos, "reward_prob"])
        # Hand-tuned affine map: belief = baseline + (1 - baseline) * reward_prob, clipped.
        # 0.35 = minimum suspicion when reward_prob is 0 (never fully "calm" from this cue alone).
        # 0.65 = 1.0 - 0.35 so reward_prob == 1 maps to belief == 1.0; not independent constants.
        return float(np.clip(0.35 + 0.65 * base_reward, 0.0, 1.0))

    def _register_chase_clearance(self) -> None:
        """Ground truth: mark alien cells near security as cleared after a chase_at_spot."""
        r0 = int(self.cfg.chase_clear_radius)
        for (r, c), row in self.env.iterrows():
            if int(row["alien_level"]) <= 0:
                continue
            if self._manhattan(self.pos, (r, c)) <= r0:
                self.alien_cells_cleared.add((r, c))

    def _gold_mines_around(self, center: Tuple[int, int], radius: int = 2) -> int:
        cr, cc = center
        n = 0
        for (r, c), row in self.env.iterrows():
            if abs(r - cr) + abs(c - cc) <= radius and str(row["mine_type"]).strip() != "":
                n += 1
        return n

    def compute_vtype(
        self,
        forager_pos: Tuple[int, int],
        revealed_count: int,
        total_cells: int,
        stunned_count_total: int,
        global_step: int,
    ) -> Dict[str, float]:
        """
        Task-type signal: high when forager is far, map mostly revealed, and stuns are rare.
        Used inside Vtype_t and downstream Vmove_* terms.
        """
        distance_t = self._manhattan(forager_pos, self.pos)
        distance_norm_t = float(np.clip(distance_t / self.Dmax, 0.0, 1.0))
        area_revealed_t = float(np.clip(revealed_count / max(1, total_cells), 0.0, 1.0))
        stun_rate_t = float(np.clip(stunned_count_total / max(1, global_step + 1), 0.0, 1.0))
        vtype_t = distance_norm_t * area_revealed_t * (1.0 - stun_rate_t)
        vtype_t = float(np.clip(vtype_t, 0.0, 1.0))
        return {
            "distance_t": distance_t,
            "distance_norm_t": distance_norm_t,
            "area_revealed_t": area_revealed_t,
            "stun_rate_t": stun_rate_t,
            "vtype_t": vtype_t,
        }

    def _role_move_one_step(
        self,
        forager_pos: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
        exclude: Optional[Tuple[int, int]],
        final_role: str,
    ) -> Tuple[bool, str]:
        """One lead (predict-ahead) or follow (orbit) step; shared by role beats and chase-skipped."""
        if final_role == "lead":
            cur = forager_pos
            if forager_pos_before is not None:
                dr = cur[0] - forager_pos_before[0]
                dc = cur[1] - forager_pos_before[1]
                predicted = (cur[0] + dr, cur[1] + dc)
                if predicted not in self.env.index:
                    predicted = cur
            else:
                predicted = cur
            moved = self._step_toward(predicted, exclude=exclude)
            return moved, "move_lead" if moved else "lead_no_step"
        moved = self._orbit_near(forager_pos, self.cfg.follow_radius, exclude=exclude)
        return moved, "move_follow" if moved else "follow_no_step"

    def _update_weights(self):
        # Nudge lead_weight / follow_weight toward whichever Vmove branch averaged higher recently.
        if not self.vmove_lead_hist or not self.vmove_follow_hist:
            return
        avg_lead = float(np.mean(self.vmove_lead_hist))
        avg_follow = float(np.mean(self.vmove_follow_hist))
        g = self.cfg.gamma
        if avg_lead > avg_follow:
            self.lead_weight = min(1.0, self.lead_weight + g * (avg_lead - avg_follow))
            self.follow_weight = max(0.0, 1.0 - self.lead_weight)
        elif avg_follow > avg_lead:
            self.follow_weight = min(1.0, self.follow_weight + g * (avg_follow - avg_lead))
            self.lead_weight = max(0.0, 1.0 - self.follow_weight)

    def step(
        self,
        t_in_round: int,
        forager_agent: ForagerAgent,
        round_idx: int,
        global_step: int,
        forager_pos_start: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
    ) -> Tuple[str, float]:
        """
        Security’s turn only (forager frozen). forager_pos_start = current forager cell;
        forager_pos_before = forager cell before their last move (velocity hint for lead).

        If ``forager_agent.stunned_steps > 0`` (alien stun lockout), overrides beats and moves
        toward the forager each step (see class docstring).
        """
        prev_before_move = tuple(self.pos)
        exclude = self._prev_security_pos

        forager_pos = forager_pos_start
        revealed_count = len(forager_agent.visited)
        total_cells = len(self.env.index)
        stunned_total = sum(1 for x in forager_agent.frames_action if x == "stunned")

        m = self.compute_vtype(forager_pos, revealed_count, total_cells, stunned_total, global_step)
        vtype_t = m["vtype_t"]
        role_by_vtype = "lead" if vtype_t >= self.cfg.vtype_threshold else "follow"

        p_belief_here = self._alien_belief_at(self.pos)
        gold_mines_around = self._gold_mines_around(forager_pos, radius=2)
        inferred_forager_movement = gold_mines_around * (1.0 - p_belief_here)
        vmove_base = inferred_forager_movement * p_belief_here * vtype_t

        lead_bonus = self.lead_weight * (1.0 + m["distance_norm_t"])
        follow_bonus = self.follow_weight * (1.0 + (1.0 - m["distance_norm_t"]))
        vmove_lead_t = vmove_base * lead_bonus
        vmove_follow_t = vmove_base * follow_bonus

        self.vmove_lead_hist.append(vmove_lead_t)
        self.vmove_follow_hist.append(vmove_follow_t)
        self._update_weights()

        if vmove_lead_t > vmove_follow_t:
            role_by_choice = "lead"
        elif vmove_follow_t > vmove_lead_t:
            role_by_choice = "follow"
        else:
            role_by_choice = self.cfg.strategy_if_tie

        # Near-tie falls back to Vtype-based role; clear winner uses role_by_choice.
        final_role = role_by_choice if abs(vmove_lead_t - vmove_follow_t) > 1e-9 else role_by_vtype

        # Pin strategic role for ablations: overrides the Vmove / Vtype choice above.
        role_forced = False
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            final_role, role_forced = "lead", True
        elif rm in ("follow", "follower"):
            final_role, role_forced = "follow", True

        p_belief = self._alien_belief_at(self.pos)
        beat_role_move = self._next_beat_is_role_move

        moved = False
        move_action = "idle"
        rescue_stunned = forager_agent.stunned_steps > 0
        if rescue_stunned:
            moved = self._step_toward(forager_pos, exclude=exclude)
            move_action = "move_rescue_stunned" if moved else "rescue_stunned_no_step"
        elif beat_role_move:
            self._next_beat_is_role_move = False
            moved, move_action = self._role_move_one_step(
                forager_pos, forager_pos_before, exclude, final_role
            )
        else:
            self._next_beat_is_role_move = True
            thr = float(self.cfg.chase_belief_threshold)
            if p_belief >= thr:
                moved = False
                move_action = "chase_at_spot"
                self._register_chase_clearance()
            else:
                moved, move_action = self._role_move_one_step(
                    forager_pos, forager_pos_before, exclude, final_role
                )

        if (
            not moved
            and self.pos != forager_pos
            and move_action != "chase_at_spot"
            and move_action != "rescue_stunned_no_step"
        ):
            if self._step_toward(forager_pos, exclude=exclude):
                move_action = move_action + "+fallback_to_forager"
                moved = True

        # Scan AFTER step (belief-only; no hidden alien_id).
        p_belief_after = self._alien_belief_at(self.pos)
        vscan_t = self.cfg.e_scan * p_belief_after

        delta_t = float(np.clip(self.lead_weight - self.follow_weight, 0.0, 1.0))

        self._prev_security_pos = prev_before_move

        self.log.append({
            "global_step": global_step,
            "round": round_idx,
            "step_in_round": t_in_round,
            "security_row": self.pos[0],
            "security_col": self.pos[1],
            "security_move": move_action,
            "security_role": final_role,
            "role_forced": role_forced,
            "rescue_stunned": rescue_stunned,
            "security_beat_role_move": beat_role_move,
            "p_alien_belief": p_belief,
            "chase_belief_threshold": float(self.cfg.chase_belief_threshold),
            "Vscan_t": vscan_t,
            "distance_t": m["distance_t"],
            "distance_norm_t": m["distance_norm_t"],
            "area_revealed_t": m["area_revealed_t"],
            "stun_rate_t": m["stun_rate_t"],
            "Vtype_t": vtype_t,
            "Vmove_lead_t": vmove_lead_t,
            "Vmove_follow_t": vmove_follow_t,
            "delta_t": delta_t,
            "lead_weight": self.lead_weight,
            "follow_weight": self.follow_weight,
        })

        return move_action, vscan_t


# =============================================================================
# Frame buffer + run
# =============================================================================
@dataclass
class AnimFrame:
    turn_idx: int  # which alternating round: 0 .. num_turn_rounds-1; -1 on start frame only
    phase: str  # "forager" | "security" | "—" (start frame only)
    step_in_phase: int  # 0 .. moves_per_round-1 within the active agent’s block; -1 on start frame
    global_step: int
    forager_pos: Tuple[int, int]
    security_pos: Tuple[int, int]
    forager_action: str
    security_move: str
    security_role: str
    vscan: float
    depleted: Set[Tuple[int, int]]  # mines with zero remaining digs (shown as NaN / gray in GIF)
    forager_total_reward: float
    forager_round_reward: float
    is_start: bool = False


def run_foraging_simulation(
    csv_path: str,
    rounds: int = 3,
    forager_cfg: Optional[ForagerConfig] = None,
    security_cfg: Optional[SecurityConfig] = None,
    seed: int = 0,
    out_prefix: str = "forager",
    save_csv: bool = True,
    save_gif: bool = True,
    forager_identity: str = "auto",
    security_identity: str = "auto",
) -> Dict[str, object]:
    """
    `rounds` counts alternating 5-step blocks (see `moves_per_round`): even indices are forager’s
    turn, odd indices are security’s turn. Example `rounds=4` → forager, security, forager, security.
    Total primitive steps = `rounds * moves_per_round` (plus one initial GIF frame).

    Identities ( **`forager_identity`**, **`security_identity`** )
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Each must be ``"auto"``, ``"leader"``, or ``"follower"`` (or aliases accepted by
    `_normalize_identity`). They are normalized and written into the configs with
    ``replace(forager_cfg, role_mode=...)`` and ``replace(security_cfg, role_mode=...)``, so they
    **replace** any ``role_mode`` you already put on ``ForagerConfig`` / ``SecurityConfig``.

    **Example — forager acts as leader, security as dedicated follower:**

        run_foraging_simulation(
            csv_path,
            rounds=20,
            forager_identity="leader",
            security_identity="follower",
        )

    Effect: forager moves prefer **distance from security** and **higher exploration weight**;
    security **follows** using lead/follow formulas; ``chase_at_spot`` uses belief vs a threshold,
    not direct alien visibility (see ``SecurityAgent`` docstring).

    **auto** keeps both agents on the adaptive rules in code (forager: your ``lambda_sec`` and
    logistic ``w_t`` only; security: dynamic lead vs follow from ``Vmove_lead_t`` / ``Vmove_follow_t``
    and ``Vtype_t``).
    """
    env = load_env_from_csv(csv_path)
    forager_cfg = forager_cfg or ForagerConfig(seed=seed)
    security_cfg = security_cfg or SecurityConfig(seed=seed)

    fi = _normalize_identity(forager_identity)
    si = _normalize_identity(security_identity)
    # Identity kwargs win over any role_mode already set on the passed-in dataclass instances.
    forager_cfg = replace(forager_cfg, role_mode=fi)
    security_cfg = replace(security_cfg, role_mode=si)

    max_r = int(env.index.get_level_values(0).max())
    max_c = int(env.index.get_level_values(1).max())
    start = (max_r // 2, max_c // 2)

    forager = ForagerAgent(env, forager_cfg)
    security = SecurityAgent(env, security_cfg, start_pos=start)
    forager.security_pos = security.pos
    forager.security_cleared_alien_cells = security.alien_cells_cleared

    frames: List[AnimFrame] = []
    moves = forager_cfg.moves_per_round
    num_turn_rounds = rounds
    global_step = 0

    # Opening frame: both agents visible before any move
    frames.append(
        AnimFrame(
            turn_idx=-1,
            phase="—",
            step_in_phase=-1,
            global_step=-1,
            forager_pos=tuple(forager.pos),
            security_pos=tuple(security.pos),
            forager_action="(start)",
            security_move="(start)",
            security_role="—",
            vscan=0.0,
            depleted=set(),
            forager_total_reward=forager.total_reward,
            forager_round_reward=forager.round_reward,
            is_start=True,
        )
    )

    # Each value of `rounds` is ONE 5-step block by ONE agent; even idx → forager, odd → security
    for turn_idx in range(num_turn_rounds):
        forager_turn = (turn_idx % 2 == 0)
        forager.current_round = turn_idx

        if forager_turn:
            for k in range(moves):
                if k == 0 and turn_idx > 0:
                    forager.reset_segment_reward()
                forager.security_pos = tuple(security.pos)
                fa = forager.step()
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="forager",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),
                        security_pos=tuple(security.pos),
                        forager_action=fa,
                        security_move="(frozen)",
                        security_role="—",
                        vscan=0.0,
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        is_start=False,
                    )
                )
                global_step += 1
        else:
            for k in range(moves):
                forager_pos_start = tuple(forager.pos)
                forager_prev = forager.prev_pos
                security.step(
                    t_in_round=k,
                    forager_agent=forager,
                    round_idx=turn_idx,
                    global_step=global_step,
                    forager_pos_start=forager_pos_start,
                    forager_pos_before=forager_prev,
                )
                sec_row = security.log[-1]
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="security",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),
                        security_pos=tuple(security.pos),
                        forager_action="(frozen)",
                        security_move=str(sec_row["security_move"]),
                        security_role=str(sec_row["security_role"]),
                        vscan=float(sec_row["Vscan_t"]),
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        is_start=False,
                    )
                )
                global_step += 1

    forager_log = pd.DataFrame(forager.log)
    security_log = pd.DataFrame(security.log)

    if save_csv:
        forager_log.to_csv(f"{out_prefix}_forager.csv", index=False)
        security_log.to_csv(f"{out_prefix}_security.csv", index=False)

    if save_gif:
        animate_simulation(
            env,
            frames,
            num_turn_rounds=rounds,
            moves_per_round=forager_cfg.moves_per_round,
            outpath=f"{out_prefix}.gif",
        )

    return {
        "env": env,
        "forager": forager,
        "security": security,
        "forager_log": forager_log,
        "security_log": security_log,
        "frames": frames,
        "total_reward": forager.total_reward,
        "rounds": rounds,
        "moves_per_round": forager_cfg.moves_per_round,
        "total_agent_steps": global_step,
        "forager_identity": fi,
        "security_identity": si,
    }


def animate_simulation(
    env: pd.DataFrame,
    frames: List[AnimFrame],
    num_turn_rounds: int,
    moves_per_round: int,
    outpath: str = "forager.gif",
):
    """Render heatmap of reward_prob; animate agents and depleted mines; save GIF via Pillow."""
    env_idx = env.set_index(["Row", "Col"], drop=False)
    max_r, max_c = int(env_idx["Row"].max()), int(env_idx["Col"].max())

    base = np.zeros((max_r + 1, max_c + 1))
    for (r, c), row in env_idx.iterrows():
        base[r, c] = row["reward_prob"]

    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(color="lightgray")
    im = ax.imshow(base, cmap=cmap, origin="upper", vmin=0, vmax=1)

    fd, = ax.plot(
        [], [],
        "o",
        color="tab:blue",
        markersize=11,
        markeredgecolor="white",
        markeredgewidth=1.5,
        label="Forager",
        zorder=5,
    )
    sd, = ax.plot(
        [], [],
        "s",
        color="limegreen",
        markersize=10,
        markeredgecolor="black",
        markeredgewidth=1.2,
        label="Security",
        zorder=6,
    )

    ax.set_xticks(range(max_c + 1))
    ax.set_yticks(range(max_r + 1))
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.legend(loc="upper right")

    def init():
        im.set_data(base)
        if frames:
            f0 = frames[0]
            r0, c0 = f0.forager_pos
            rs, cs = f0.security_pos
            if (r0, c0) == (rs, cs):
                r0p, rsp = r0 - 0.22, rs + 0.22
            else:
                r0p, rsp = r0, rs
            fd.set_data([c0], [r0p])
            sd.set_data([cs], [rsp])
        else:
            fd.set_data([], [])
            sd.set_data([], [])
        ax.set_title("")
        return [im, fd, sd]

    def update(i):
        fr = frames[i]
        g = base.copy()
        for dr, dc in fr.depleted:
            g[dr, dc] = np.nan
        im.set_data(g)

        frg, fcg = fr.forager_pos
        sr, sc = fr.security_pos
        # Same cell: separate markers slightly so security is not hidden under the forager
        if (frg, fcg) == (sr, sc):
            frg_plot, sr_plot = frg - 0.22, sr + 0.22
        else:
            frg_plot, sr_plot = frg, sr
        fd.set_data([fcg], [frg_plot])
        sd.set_data([sc], [sr_plot])

        gstep = "—" if fr.global_step < 0 else str(fr.global_step + 1)
        stext = "—" if fr.step_in_phase < 0 else f"{fr.step_in_phase + 1}/{moves_per_round}"

        if fr.is_start:
            title = (
                f"Alternating rounds: each round = {moves_per_round} steps by ONE agent (no world reset)\n"
                f"INITIAL  |  global step {gstep}  |  next: ROUND 1 = FORAGER ×{moves_per_round}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}"
            )
        else:
            if fr.phase == "forager":
                active = "FORAGER moving"
                sec_extra = ""
            elif fr.phase == "security":
                active = "SECURITY moving"
                sec_extra = f" ({fr.security_role})  |  Vscan={fr.vscan:.3f}"
            else:
                active = str(fr.phase)
                sec_extra = ""
            frozen_note = (
                "other agent frozen"
                if fr.phase in ("forager", "security")
                else ""
            )
            title = (
                f"ROUND {fr.turn_idx + 1}/{num_turn_rounds}  |  {active}  |  step {stext}  |  global {gstep}  |  {frozen_note}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}{sec_extra}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}"
            )
        ax.set_title(title, fontsize=9)
        return [im, fd, sd]

    # blit=False avoids dots disappearing with changing title / im data
    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(frames),
        interval=450,
        blit=False,
        repeat=True,
    )
    anim.save(outpath, writer=PillowWriter(fps=3))
    plt.close(fig)



In [54]:

if __name__ == "__main__":
    single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
    if not os.path.isfile(single_csv):
        print(f"Skip run: CSV not found: {single_csv}")
    else:
        out = run_foraging_simulation(
            single_csv,
            rounds=20,
            forager_cfg=ForagerConfig(moves_per_round=5, seed=0),
            security_cfg=SecurityConfig(seed=0),
            # Example fixed roles: forager_identity="leader", security_identity="follower"
            out_prefix="forager_sim",
        )
        print("Total reward:", out["total_reward"])
        print("Frames:", len(out["frames"]))


Total reward: 14.299999999999999
Frames: 101
